<a href="https://colab.research.google.com/github/zuicagerman-eng/Coloso_Holcim/blob/claude%2Fholcim-github-vs-google-script-kxgrv0/convertir_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [51]:
def cargar_datos():
  from google.colab import auth
  import gspread
  from google.auth import default
  import pandas as pd
  import numpy as np

  auth.authenticate_user()
  creds, _=default()
  gc=gspread.authorize(creds)

  sh=gc.open("Datos_tablero_HS")
  worksheet = sh.worksheet("datos")
  df, df_resumen, df_resumen_completo,_=nuevos_datos()
  valores=df.astype(str).values.tolist()
  titulos=[df.columns.tolist()]
  worksheet.clear()
  worksheet.append_rows(titulos)
  worksheet.append_rows(valores, table_range="A2")

  worksheet_r = sh.worksheet("resumen")
  valores_r=df_resumen.astype(str).values.tolist()
  titulos_r=[df_resumen.columns.tolist()]
  worksheet_r.clear()
  worksheet_r.append_rows(titulos_r)
  worksheet_r.append_rows(valores_r)

  worksheet_r_c = sh.worksheet("resumen_completo")
  valores_r_c=df_resumen_completo.astype(str).values.tolist()
  titulos_r_c=[df_resumen_completo.columns.tolist()]
  worksheet_r_c.clear()
  worksheet_r_c.append_rows(titulos_r_c)
  worksheet_r_c.append_rows(valores_r_c)

  df_a_capacitar_total=tabla_capacitaciones()
  worksheet_t_c = sh.worksheet("tabla capacitaciones")
  valores_t_c=df_a_capacitar_total.astype(str).values.tolist()
  titulos_t_c=[df_a_capacitar_total.columns.tolist()]
  worksheet_t_c.clear()
  worksheet_t_c.append_rows(titulos_t_c)
  worksheet_t_c.append_rows(valores_t_c, table_range="A2")

  df_presupuesto=seguimiento_presupuesto()
  worksheet_p = sh.worksheet("presupuesto")
  valores_p=df_presupuesto.astype(str).values.tolist()
  titulos_p=[df_presupuesto.columns.tolist()]
  worksheet_p.clear()
  worksheet_p.append_rows(titulos_p)
  worksheet_p.append_rows(valores_p, table_range="A2")





In [52]:
def nuevos_datos():
  from google.colab import auth
  import gspread
  from google.auth import default
  import pandas as pd
  from datetime import date

  #Cálculo de fecha para diferentes operaciones
  hoy=date.today()
  anio_proximo = date.today().year+1

  #Autenticación en google para poder leer y escribir libros de google sheets
  auth.authenticate_user()
  creds, _=default()
  gc=gspread.authorize(creds)

  #Cargar datos del libro de matriz consolidada y convertirlo en un DataFrame
  sh=gc.open("Matriz_HS_Consolidada")
  worksheet = sh.worksheet("m_cap_H&S")
  data=worksheet.get_all_records()
  df = pd.DataFrame(data)
  #Solo se dejan los registros donde el nombre es no vacío
  df=df[df.NOMBRE!=""]
  #Solo se dejan los registros cuyo STATUS DEL PERSONAL es ACTIVO
  df=df[df['STATUS DEL PERSONAL']=="ACTIVO"]
  #Se crea lista de las columnas del DataFrame
  columnas=list(df.columns)

  #Se trabaja con las primeras 7 columnas para generar el nuevo DataFrame (df_prueba)
  col_ext=columnas[0:6]
  df_prueba=df[col_ext]
  nombre_curso=col_ext[5]
  df_prueba=df_prueba.copy()
  #Se crea la columna soporte y se asignan los respectivos valores
  df_prueba["Soporte"]=df_prueba.iloc[:,5].apply(lambda x: "Pendiente" if x=="" else "Hipervinculado")
  #Se crea la columna CURSO/CAPACITACION y se asignan el nombre del primer curso
  df_prueba["CURSO/CAPACITACION"]=nombre_curso
  #Se crea la columna ESTADO, inicialmente vacía
  df_prueba["ESTADO"]=""
  #Se renombra la columna que tiene el nombre del curso por Fecha
  df_prueba.rename(columns={nombre_curso:"Fecha"}, inplace=True)
  #Se crea máscara y en caso que no exita fecha la columna ESTADO se le da el valor Pendiente
  mask_vacio = df_prueba["Fecha"] == ""
  df_prueba.loc[mask_vacio, "ESTADO"] = "Pendiente"

  #Se va a poblar la información de los 61 cursos restantes
  f=7
  c=9
  for l in range(61):#61
    #Se toman las 6 primeras columnas que son fijas para todos los cursos
    col_ext=columnas[0:5]
    #Se toman dinámicamente el resto de columnas para cada curso
    col_nvas=columnas[f:c]
    col_ext=col_ext+col_nvas
    #Se genera un nuevo DataFrame (df_nuevo) con las columnas respectivas
    df_nuevo=df[col_ext]
    df_nuevo=df_nuevo.copy()
    nombre_curso=col_ext[5]
    #La columna 6 del nuevo DataFrame se renombre a Soporte
    df_nuevo.rename(
      columns={df_nuevo.columns[6]: "Soporte"},
      inplace=True
      )
    #La Columna que tiene el nombre del curso se renombra a Fecha
    df_nuevo["ESTADO"]=df_nuevo[nombre_curso]
    df_nuevo.rename(columns={nombre_curso:"Fecha"}, inplace=True)
    #Se crea una máscara para cambiar el estado para los campos que tienen una fecha que el año es superior a 2100. EL nuevo estado es Vitalicio
    mask = (
    pd.to_numeric(
        df_nuevo["ESTADO"].astype(str).str[-4:],
        errors="coerce"
      ) > 2100
    )
    df_nuevo.loc[mask, "ESTADO"] = "Vitalicio"
    #Si el campo Fecha no tiene dígitos se da valor vacío
    df_nuevo.loc[
      ~df_nuevo["Fecha"].str.contains(r"\d", regex=True, na=False),
      "Fecha"
      ] =""
    #Para el campo fecha se deja vacío si está vacío y se asigna a vacío si el año es mayor a 2100
    df_nuevo['Fecha']=df_nuevo.Fecha.apply(lambda x: "" if x=="" else ("" if int(x[-4:])>2100 else x))
    #Se genera el campo CURSO/CAPACITACION y se da como valores el nombre del curso en cada iteración
    df_nuevo["CURSO/CAPACITACION"]=nombre_curso
    #Concateno los DataFrame
    df_prueba=pd.concat([df_prueba,df_nuevo])
    f+=2
    c+=2
    df_prueba=df_prueba.reset_index(drop=True)
  #Ya fuera del For, y con el DataFrame completo se remplaza en ESTADO por vacío aquellos que tiene números
  df_prueba.loc[
      df_prueba["ESTADO"].str.contains(r"\d", regex=True, na=False),
      "ESTADO"
      ] =""
  #Para las fecha comodines se establece como fecha del sistema 31/12/2100 y se cambia el formato a fecha
  df_prueba['Fecha']=df_prueba['Fecha'].apply(lambda x: "31/12/2100" if x=="31/12/9999" else
                                                ("31/12/2100" if x=="30/12/9999"
                                                else ("31/12/2100" if x=="20/12/9999" else ("21/11/2026" if x=="21/11/5026" else x))))

  df_prueba['Fecha']= pd.to_datetime(df_prueba['Fecha'], format='%d/%m/%Y', errors="coerce")
  #Se calcula la diferencia entre el día de hoy y la fecha de vencimiento, para seguir clasificando
  df_prueba["dif_fecha"]=df_prueba['Fecha']-pd.to_datetime(hoy)
  df_prueba["dif_fecha"]=df_prueba["dif_fecha"]/pd.Timedelta(days=1)
  # Se crean los Estados Vigente y Vencido, se da formato al tetxo de Soporte y Estado y se crea una nueva columna de estado_definitivo
  mask_vigente = df_prueba["dif_fecha"] >= 0
  mask_vencido = df_prueba["dif_fecha"] < 0
  df_prueba.loc[mask_vigente, "ESTADO"] = "Vigente"
  df_prueba.loc[mask_vencido, "ESTADO"] = "Vencido"
  df_prueba['ESTADO']=df_prueba['ESTADO'].apply(lambda x: x.capitalize())
  df_prueba['estado_definitivo']=df_prueba['ESTADO']
  df_prueba['Soporte']=df_prueba['Soporte'].apply(lambda x: x.capitalize())

 #Se excluyen las siguientes capacitaciones Examanes medicos,Polipasto <5 toneladas, Aparejador/señalero, Puente grua / Polipasto
  excluir=['EXAMEN MEDICO  OCUPACIONAL',' Aparejador/señalero', ' Polipasto < 5 Toneladas',' Puente grua / Polipasto']
  df_prueba=df_prueba[~df_prueba['CURSO/CAPACITACION'].isin(excluir)]

  #Se agregan dos columnas al DataFrame
  coulmnas_extra=["NOMBRE","Grupo de personal"]
  df_extra=df[coulmnas_extra]
  df_completa=df_prueba.copy()
  df_completa=df_completa.merge(df_extra, on="NOMBRE")

  #Se crea un nuevo DataFrame con los registros del año actual y anteriores y los que tienen la fecha vacía
  df_com_anio=df_completa[(df_completa.Fecha.dt.year<anio_proximo)|(df_completa.Fecha.isna())]
  #Se crea un nuevo DataFrame excluyendo unos estados y manteniendo solo unos grupos
  estados=["Exceptuado","No aplica", "Vitalicio"]
  df_com_anio_estados=df_com_anio[~df_com_anio.ESTADO.isin(estados)]
  grupo=["Activos","Externos"]
  df_com_anio_estados_gr=df_com_anio_estados[df_com_anio_estados["Grupo de personal"].isin(grupo)].copy()
  df_com_anio_estados_gr['CURSO/CAPACITACION']=df_com_anio_estados_gr['CURSO/CAPACITACION'].apply(lambda x: x.strip()).copy()

  #Se crean dos DataFrame de salida
  tabla_resumen=df_com_anio_estados_gr.groupby(['CURSO/CAPACITACION', 'Grupo de personal','DIVISIÓN'])["NOMBRE"].count().reset_index(name='COUNT')
  df_prueba=df_prueba[['CODIGO ','ID','DIVISIÓN','NOMBRE','POSICIÓN','ESTADO','CURSO/CAPACITACION','Fecha',"estado_definitivo","Soporte"]]

  #Importar hoja de tarifas

  ws_tarifas=sh.worksheet("Tarifas")
  data_tarifas=ws_tarifas.get_all_records()
  df_tarifas = pd.DataFrame(data_tarifas)
  df_tarifas['COSTO CAPACITACION']=df_tarifas['COSTO CAPACITACION'].apply(lambda x: x.replace("$",""))
  df_tarifas['COSTO CAPACITACION']=df_tarifas['COSTO CAPACITACION'].apply(lambda x: x.replace(".",""))
  df_tarifas['COSTO CAPACITACION']=df_tarifas['COSTO CAPACITACION'].astype(int)
  df_tarifas=df_tarifas.rename(columns={"":"LLAVE"})

  #Calcular el mes de la fecha de vnecimiento y generar la tabla de resumen con mes
  df_resumen_m=df_com_anio_estados_gr.copy()
  df_resumen_m['Fecha']=df_resumen_m['Fecha'].apply(lambda x: hoy if pd.isna(x) else x)
  df_resumen_m['Fecha']= pd.to_datetime(df_resumen_m['Fecha'], format='%d/%m/%Y', errors="coerce")
  df_resumen_m["MES"]=df_resumen_m["Fecha"].dt.month
  df_resumen_m["AÑO"]=df_resumen_m["Fecha"].dt.year
  tabla_resumen_m=df_resumen_m.groupby(['CURSO/CAPACITACION', 'Grupo de personal','DIVISIÓN','AÑO','MES'], dropna=False)["NOMBRE"].count().reset_index(name='COUNT').copy()
  tabla_resumen_m["LLAVE"]=tabla_resumen_m["CURSO/CAPACITACION"]+tabla_resumen_m["Grupo de personal"]+tabla_resumen_m["DIVISIÓN"]

  #tabla_resumen_m=tabla_resumen_m.rename(columns={"CURSO/CAPACITACION":"CAPACITACION","DIVISIÓN":"DIVISION"})
  tabla_resumen_completa=pd.merge(tabla_resumen_m,df_tarifas[["LLAVE","COSTO CAPACITACION"]], on="LLAVE", how="left")
  tabla_resumen_completa["COSTO CAPACITACION"]=tabla_resumen_completa["COSTO CAPACITACION"].fillna(0)
  tabla_resumen_completa["COSTO CAPACITACION"]=tabla_resumen_completa["COSTO CAPACITACION"].apply(lambda x: int(x))
  tabla_resumen_completa["Fecha_completa"]=pd.to_datetime(tabla_resumen_completa["AÑO"].astype(str)+"-"+tabla_resumen_completa["MES"].astype(str)+"-01", format="%Y-%m-%d")

  #Se genera un nuevo DF de las personas que deben capacitarse por mes y año, según el vencimiento de la capacitación
  año_act=hoy.year
  mes_act=hoy.month
  import numpy as np

  df_prueba["año_vencimiento"] = np.where(
      df_prueba["estado_definitivo"] == "Pendiente",
      año_act,
      df_prueba["Fecha"].dt.year
  )

  df_prueba["mes_vencimiento"] = np.where(
      df_prueba["estado_definitivo"] == "Pendiente",
      mes_act,
      df_prueba["Fecha"].dt.month
  )

  df_a_capacitar=df_prueba.groupby(["año_vencimiento", "mes_vencimiento", "DIVISIÓN",
                    "CURSO/CAPACITACION"])["CODIGO "].count().reset_index()


  return df_prueba, tabla_resumen,tabla_resumen_completa,df_a_capacitar

In [53]:
def cronograma_capacitaciones():
  from google.colab import auth
  import gspread
  from google.auth import default
  import pandas as pd
  from datetime import date

  fecha_control = pd.Timestamp("1900-01-01")
  #Cálculo de fecha para diferentes operaciones
  hoy=date.today()
  anio_proximo = date.today().year+1

  #Autenticación en google para poder leer y escribir libros de google sheets
  auth.authenticate_user()
  creds, _=default()
  gc=gspread.authorize(creds)

  #Cargar datos de PROGRAMACION del libro de matriz consolidada y convertirlo en un DataFrame
  sh=gc.open("Matriz_HS_Consolidada")
  worksheet = sh.worksheet("PROGRAMACION")
  data=worksheet.get_all_records()
  df = pd.DataFrame(data)

  #Se eliminan los registros donde INREGISTRO este vacío
  df=df[df.IDREGISTRO!=""]
  #Se crea una llave con el fin de solo dejar las últimas capacitaciones por persona y se borran los duplicados
  df["llave"]=df.NOMBRE+df.CAPACITACIONES
  df["llave"] = (
      df["llave"]
      .astype(str)
      .str.replace("\xa0", "", regex=False)  # espacio no separable
      .str.strip()
      .str.lower()
  )
  #df=df.drop_duplicates(subset=["llave"], keep="last")
  #Limpieza de datos
  df["NOMBRE"]=df["NOMBRE"].apply(lambda x: "SIN INFORMACIÓN" if x== '' else x)
  df["DIVISIÓN"]=df["DIVISIÓN"].apply(lambda x: "SIN INFORMACIÓN" if x== '' else x)
  df["DIVISIÓN"]=df["DIVISIÓN"].apply(lambda x: "SIN INFORMACIÓN" if x== '#N/A' else x)
  df["CAPACITACIONES"]=df["CAPACITACIONES"].apply(lambda x: "SIN INFORMACIÓN" if x== '' else x)
  df["FECHA DE VENCIMIENTO"]=df["FECHA DE VENCIMIENTO"].apply(lambda x: "SIN INFORMACIÓN" if x== '' else x)
  df["PROVEEDOR"]=df["PROVEEDOR"].apply(lambda x: "SIN INFORMACIÓN" if x== '' else x)
  df["ASISTENCIA"]=df["ASISTENCIA"].apply(lambda x: "POR CONFIRMAR" if x== '' else x)
  df["CODIGO "]=df["CODIGO "].apply(lambda x: "POR CONFIRMAR" if x== '' else x)
  df["ID"]=df["ID"].apply(lambda x: "SIN INFORMACIÓN" if x== '' else x)
  df["ID"]=df["ID"].apply(lambda x: "SIN INFORMACIÓN" if x== '#N/A' else x)
  df["POSICIÓN"]=df["POSICIÓN"].apply(lambda x: "SIN INFORMACIÓN" if x== '' else x)
  df["POSICIÓN"]=df["POSICIÓN"].apply(lambda x: "SIN INFORMACIÓN" if x== '#N/A' else x)
  df["FECHA DE ASISTENCIA"]=df["FECHA DE ASISTENCIA"].apply(lambda x: fecha_control if x== 'SIN INFORMACION' else x)
  df["FECHA DE ASISTENCIA"]=df["FECHA DE ASISTENCIA"].apply(lambda x: fecha_control if x== 'SIN INFORMACIÓN' else x)
  df["FECHA DE ASISTENCIA"]=df["FECHA DE ASISTENCIA"].apply(lambda x: fecha_control if x== '' else x)
  df["FECHA DE ASISTENCIA"]=df["FECHA DE ASISTENCIA"].apply(lambda x: fecha_control if x== '' else x)
  df["VIGENCIA"]=df["VIGENCIA"].apply(lambda x: 0 if x== '' else x)
  df["VIGENCIA"]=df["VIGENCIA"].apply(lambda x: 0 if x== '#N/A' else x)

  #Se da forma de fecha a FECHA DE ASISTENCIA y se crena las variables de año y mes
  df['FECHA DE ASISTENCIA']= pd.to_datetime(df['FECHA DE ASISTENCIA'], format='%d/%m/%Y', errors="coerce")
  df["año_asistencia"]=df["FECHA DE ASISTENCIA"].dt.year
  df["mes_asistencia"]=df["FECHA DE ASISTENCIA"].dt.month
  #Se genera el DF de programados y un df más con la asistencia a las capacitaciones


  df_programados=df.groupby(["año_asistencia", "mes_asistencia", "DIVISIÓN",
                    "CAPACITACIONES"])["IDREGISTRO"].count().reset_index()
  df_programados_s=df.groupby(["año_asistencia", "mes_asistencia", "DIVISIÓN",
                    "CAPACITACIONES","ASISTENCIA"])["IDREGISTRO"].count().reset_index()

  return df_programados, df_programados_s

In [54]:
#Se crea una función para formatear los datos y que permitan hacer cruces entre DF
def formateo_texto(df, campo):
  df[campo]=(
      df[campo]
      .astype(str)
      .str.replace("\xa0", "", regex=False)  # espacio no separable
      .str.strip()
      .str.lower()
  )


In [55]:
def tabla_capacitaciones():
  import pandas as pd
  #Ejecuto funciones para obtener los df necesarios
  _, _, _, df_a_capacitar=nuevos_datos()
  df_programados,df_programados_s =cronograma_capacitaciones()

  df_programados_s=df_programados_s[df_programados_s.año_asistencia>2017]
  df_programados=df_programados[df_programados.año_asistencia>2017]

  #Renombro los campos de los DF para poder hacer el merge entre ellos
  df_a_capacitar=df_a_capacitar.rename(columns={"año_vencimiento":"año", "mes_vencimiento":"mes", "DIVISIÓN":"Division",
                                      "CURSO/CAPACITACION":"Capacitaciones","CODIGO ":"A_capacitar"})

  df_programados=df_programados.rename(columns={"año_asistencia":"año", "mes_asistencia":"mes", "DIVISIÓN":"Division",
                                      "CAPACITACIONES":"Capacitaciones","IDREGISTRO":"Programados"})
  df_programados_s=df_programados_s.rename(columns={"año_asistencia":"año", "mes_asistencia":"mes", "DIVISIÓN":"Division",
                                      "CAPACITACIONES":"Capacitaciones","IDREGISTRO":"Programados"})
  #Aplico el formato a las columnas necesarias
  formateo_texto(df_a_capacitar, "Division")
  formateo_texto(df_a_capacitar, "Capacitaciones")
  formateo_texto(df_programados, "Division")
  formateo_texto(df_programados, "Capacitaciones")
  formateo_texto(df_programados_s, "Division")
  formateo_texto(df_programados_s, "Capacitaciones")
  #Cruzo df_a_capacitar_total con df_programados
  #df_a_capacitar_total =pd.merge(df_a_capacitar,df_programados, how="left",on=['año', 'mes', 'Division', 'Capacitaciones'])
  df_a_capacitar_total=pd.concat([df_a_capacitar,df_programados])
  df_a_capacitar_total=df_a_capacitar_total.groupby(["año","mes","Division","Capacitaciones"])[["A_capacitar","Programados"]].sum().reset_index()
  df_a_capacitar_total["Total"]=df_a_capacitar_total.A_capacitar+df_a_capacitar_total.Programados
  #Genero los DF con quienes asistieron, con los que no y con los que están pendientes
  df_programados_s_si=df_programados_s[df_programados_s.ASISTENCIA=="SI"].copy()
  df_programados_s_no=df_programados_s[df_programados_s.ASISTENCIA=="NO"].copy()
  df_programados_s_vacios=df_programados_s[(df_programados_s.ASISTENCIA=="POR CONFIRMAR")|(df_programados_s.ASISTENCIA=="SIN INFORMACIÓN")].copy()
  #Hago el concat para generar la columna de los que asistieron
  df_programados_s_si.rename(columns={"Programados":"Asistieron"}, inplace=True)
  df_a_capacitar_total=pd.concat([df_a_capacitar_total,df_programados_s_si])
  df_a_capacitar_total=df_a_capacitar_total.groupby(["año","mes","Division","Capacitaciones"])[["A_capacitar","Programados","Total","Asistieron"]].sum().reset_index()
  #Hago el concat para generar la columna de los que No asistieron
  df_programados_s_no.rename(columns={"Programados":"No_Asistieron"}, inplace=True)
  df_a_capacitar_total=pd.concat([df_a_capacitar_total,df_programados_s_no])
  df_a_capacitar_total=df_a_capacitar_total.groupby(["año","mes","Division","Capacitaciones"])[["A_capacitar",
                                                                                                "Programados","Total","Asistieron","No_Asistieron"]].sum().reset_index()
  #Hago el concat para generar la columna de los que Por Confirmar
  df_programados_s_vacios.rename(columns={"Programados":"Pendiente_confimacion"}, inplace=True)
  df_a_capacitar_total=pd.concat([df_a_capacitar_total,df_programados_s_vacios])
  df_a_capacitar_total=df_a_capacitar_total.groupby(["año","mes","Division","Capacitaciones"])[["A_capacitar","Programados","Total",
                                                                                                "Asistieron","No_Asistieron","Pendiente_confimacion"]].sum().reset_index()
  df_a_capacitar_total["Pendiente_programar"]=df_a_capacitar_total.Total-df_a_capacitar_total.Asistieron
  return df_a_capacitar_total

In [56]:
def seguimiento_presupuesto():
  from google.colab import auth
  import gspread
  from google.auth import default
  import pandas as pd
  from datetime import date

  fecha_control = pd.Timestamp("1900-01-01")

  #Cálculo de fecha para diferentes operaciones
  hoy=date.today()
  anio_proximo = date.today().year+1

  #Autenticación en google para poder leer y escribir libros de google sheets
  auth.authenticate_user()
  creds, _=default()
  gc=gspread.authorize(creds)

  #Cargar datos del libro de matriz consolidada y convertirlo en un DataFrame
  sh=gc.open("Matriz_HS_Consolidada")
  worksheet = sh.worksheet("Presupuesto_inicial")
  data=worksheet.get_all_records()
  df = pd.DataFrame(data)

  ws_programaciones= sh.worksheet("PROGRAMACION")
  data_programaciones=ws_programaciones.get_all_records()
  df_programaciones = pd.DataFrame(data_programaciones)
  df_programaciones=df_programaciones[df_programaciones.NOMBRE!=""]
  df_programaciones=df_programaciones[df_programaciones.ASISTENCIA=="SI"]
  df_programaciones["DIVISIÓN"]=df_programaciones.DIVISIÓN.apply(lambda x: "SIN INFORMACIÓN" if x== '#N/A' else x)
  df_programaciones["DIVISIÓN"]=df_programaciones.DIVISIÓN.apply(lambda x: "SIN INFORMACIÓN" if x== '' else x)
  df_programaciones["CAPACITACIONES"]=df_programaciones.CAPACITACIONES.apply(lambda x: "SIN INFORMACIÓN" if x== '' else x)
  df_programaciones["FECHA DE VENCIMIENTO"]=df_programaciones["FECHA DE VENCIMIENTO"].apply(lambda x: "SIN INFORMACIÓN" if x== '#VALUE!' else x)
  df_programaciones["FECHA DE VENCIMIENTO"]=df_programaciones["FECHA DE VENCIMIENTO"].apply(lambda x: "SIN INFORMACIÓN" if x== '' else x)
  df_programaciones["PROVEEDOR"]=df_programaciones["PROVEEDOR"].apply(lambda x: "SIN INFORMACIÓN" if x== '' else x)
  df_programaciones["ASISTENCIA"]=df_programaciones["ASISTENCIA"].apply(lambda x: "SIN INFORMACIÓN" if x== '' else x)
  df_programaciones["CODIGO "]=df_programaciones["CODIGO "].apply(lambda x: "SIN INFORMACIÓN" if x== '' else x)
  df_programaciones["CODIGO "]=df_programaciones["CODIGO "].apply(lambda x: "SIN INFORMACIÓN" if x== 'f' else x)
  df_programaciones["CODIGO "]=df_programaciones["CODIGO "].apply(lambda x: "SIN INFORMACIÓN" if x== '' else x)
  df_programaciones["CODIGO "]=df_programaciones["CODIGO "].apply(lambda x: "SIN INFORMACIÓN" if x== '#N/A' else x)
  df_programaciones["ID"]=df_programaciones["ID"].apply(lambda x: "SIN INFORMACIÓN" if x== '#N/A' else x)
  df_programaciones["ID"]=df_programaciones["ID"].apply(lambda x: "SIN INFORMACIÓN" if x== 'f' else x)
  df_programaciones["ID"]=df_programaciones["ID"].apply(lambda x: "SIN INFORMACIÓN" if x== '' else x)
  df_programaciones["POSICIÓN"]=df_programaciones["POSICIÓN"].apply(lambda x: "SIN INFORMACIÓN" if x== '#N/A' else x)
  df_programaciones["POSICIÓN"]=df_programaciones["POSICIÓN"].apply(lambda x: "SIN INFORMACIÓN" if x== '' else x)
  df_programaciones["FECHA DE ASISTENCIA"]=df_programaciones["FECHA DE ASISTENCIA"].apply(lambda x: fecha_control if x== 'SIN INFORMACION' else x)
  df_programaciones["FECHA DE ASISTENCIA"]=df_programaciones["FECHA DE ASISTENCIA"].apply(lambda x: fecha_control if x== 'SIN INFORMACIÓN' else x)
  df_programaciones["FECHA DE ASISTENCIA"]=df_programaciones["FECHA DE ASISTENCIA"].apply(lambda x: fecha_control if x== '' else x)
  df_programaciones["FECHA DE ASISTENCIA"]=df_programaciones["FECHA DE ASISTENCIA"].apply(lambda x: fecha_control if x== '' else x)
  df_programaciones["VIGENCIA"]=df_programaciones["VIGENCIA"].apply(lambda x: 0 if x== '' else x)
  df_programaciones["VIGENCIA"]=df_programaciones["VIGENCIA"].apply(lambda x: 0 if x== '#N/A' else x)
  df_programaciones.drop("VALOR", inplace=True, axis=1)
  df_programaciones['FECHA DE ASISTENCIA']= pd.to_datetime(df_programaciones['FECHA DE ASISTENCIA'], format='%d/%m/%Y', errors="coerce")
  df_programaciones["AÑO"]=df_programaciones["FECHA DE ASISTENCIA"].dt.year
  df_programaciones["MES"]=df_programaciones["FECHA DE ASISTENCIA"].dt.month
  df_programaciones=df_programaciones[df_programaciones.AÑO==2026]
  formateo_texto(df, "DIVISIÓN")
  formateo_texto(df, "CAPACITACIONES")
  formateo_texto(df_programaciones, "DIVISIÓN")
  formateo_texto(df_programaciones, "CAPACITACIONES")
  df_programaciones["VALOR"]=0
  df_presupuesto=pd.concat([df,df_programaciones])
  #df_presupuesto=df_presupuesto.groupby(["AÑO","MES","DIVISIÓN","CAPACITACIONES"])[["IDREGISTRO","total_a_capacitar"]].sum().reset_index()
  df_presupuesto = (
      df_presupuesto
      .groupby(["AÑO","MES","DIVISIÓN","CAPACITACIONES","VALOR"])
      .agg({
          "IDREGISTRO": "count",
          "total_a_capacitar": "sum"
          })
      .reset_index()
      )
  df_presupuesto=df_presupuesto.rename(columns={"DIVISIÓN":"DIVISION"})
  """
  TARIFAS
  """
  ws_tarifas=sh.worksheet("Tarifas")
  data_tarifas=ws_tarifas.get_all_records()
  df_tarifas = pd.DataFrame(data_tarifas)
  df_tarifas.rename(columns={"":"LLAVE"}, inplace=True)
  col_tarifas=["CAPACITACION", "DIVISION", "COSTO CAPACITACION"]
  df_tarifas=df_tarifas[col_tarifas]
  df_tarifas=df_tarifas.rename(columns={"CAPACITACION":"CAPACITACIONES"})
  formateo_texto(df_tarifas, "DIVISION")
  formateo_texto(df_tarifas, "CAPACITACIONES")
  df_tarifas.drop_duplicates(inplace=True)
  df_presupuesto=pd.merge(df_presupuesto,df_tarifas, how="left", on=["DIVISION","CAPACITACIONES"])
  df_presupuesto.rename(columns={"IDREGISTRO":"capacitados"}, inplace=True)
  df_presupuesto["COSTO CAPACITACION"]=df_presupuesto["COSTO CAPACITACION"].fillna(0)
  df_presupuesto["total_a_capacitar"]=df_presupuesto["total_a_capacitar"].astype(int)
  df_presupuesto["COSTO CAPACITACION"]=df_presupuesto["COSTO CAPACITACION"].apply(lambda x: str(x).replace("$",""))
  df_presupuesto["COSTO CAPACITACION"]=df_presupuesto["COSTO CAPACITACION"].apply(lambda x: str(x).replace(".",""))
  df_presupuesto["COSTO CAPACITACION"]=df_presupuesto["COSTO CAPACITACION"].astype(int)
  df_presupuesto["Valor_capacitados"]=df_presupuesto["COSTO CAPACITACION"]*df_presupuesto["capacitados"]
  df_presupuesto["Valor_presupuesto"]=df_presupuesto["VALOR"]*df_presupuesto["total_a_capacitar"]
  excluir=['EXAMEN MEDICO  OCUPACIONAL','aparejador/señalero', 'polipasto < 5 toneladas','puente grua / polipasto']
  df_presupuesto=df_presupuesto[~df_presupuesto['CAPACITACIONES'].isin(excluir)]
  return df_presupuesto